# 04 — Machine Learning

## 1. Objetivo

Evaluar si modelos de Machine Learning basados en árboles aportan capacidad
predictiva adicional frente a los modelos estadísticos desarrollados
previamente.

Se utilizarán principalmente dos familias:

- Random Forest, como representante de métodos de *bagging*;
- XGBoost, como representante de métodos de *gradient boosting*.

Los modelos se evaluarán sobre exactamente las mismas particiones utilizadas
por los GLM y mediante el mismo protocolo:

- desempeño out-of-sample;
- calibración;
- métricas específicas según el target;
- recuperación de las cantidades *oracle*;
- costo computacional.

El objetivo no es realizar un benchmark exhaustivo de algoritmos, sino estudiar
si el incremento de flexibilidad permite recuperar mejor la estructura
subyacente del riesgo.

## 2. Imports y configuración

In [3]:
from pathlib import Path
from time import perf_counter
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_poisson_deviance,
    mean_gamma_deviance,
    mean_tweedie_deviance,
)

from xgboost import XGBRegressor

SEED = 42

In [4]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.metrics import (
    regression_metrics,
    relative_improvement
)

from src.evaluation.calibration import (
    calibration_by_quantile
)

## 3. Carga de datos

In [5]:
PORTFOLIO_PATH = Path(
    "../data/raw/synthetic_insurance_portfolio.csv"
)

CLAIMS_PATH = Path(
    "../data/raw/synthetic_insurance_claims.csv"
)

SPLIT_PATH = Path(
    "../data/processed/train_test_split.csv"
)

df = pd.read_csv(PORTFOLIO_PATH)
df_claims = pd.read_csv(CLAIMS_PATH)
split_df = pd.read_csv(SPLIT_PATH)

In [ ]:
# Aplicamos el split ya definido
df = df.merge(
    split_df,
    on="policy_id",
    how="left"
)

df_train = df.loc[
    df["split"] == "train"
].copy()

df_test = df.loc[
    df["split"] == "test"
].copy()

In [7]:
df_claims = df_claims.merge(
    split_df,
    on="policy_id",
    how="left"
)

claims_train = df_claims.loc[
    df_claims["split"] == "train"
].copy()

claims_test = df_claims.loc[
    df_claims["split"] == "test"
].copy()

In [ ]:
# Checks
assert df["split"].notna().all()
assert df_claims["split"].notna().all()

print(
    f"Pólizas train/test: "
    f"{len(df_train):,} / {len(df_test):,}"
)

print(
    f"Siniestros train/test: "
    f"{len(claims_train):,} / {len(claims_test):,}"
)

Pólizas train/test: 8,000 / 2,000
Siniestros train/test: 1,239 / 315


## 4. Preprocesamiento.

In [9]:
NUMERIC_FEATURES = [
    "edad",
    "anios_vehiculo",
    "historial_siniestros"
]

CATEGORICAL_FEATURES = [
    "zona",
    "cobertura",
    "tipo_uso"
]

FEATURES = (
    NUMERIC_FEATURES
    + CATEGORICAL_FEATURES
)

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            CATEGORICAL_FEATURES
        )
    ],
    remainder="passthrough"
)

## 5. Frecuencia

### 5.1 Target

In [11]:
X_train_freq = df_train[FEATURES]
X_test_freq = df_test[FEATURES]

y_train_freq = df_train["numero_siniestros"]
y_test_freq = df_test["numero_siniestros"]

## 6. Random Forest para frecuencia

In [12]:
rf_frequency = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                max_depth=None,
                min_samples_leaf=5,
                random_state=SEED,
                n_jobs=-1,
                criterion="poisson"  # usar una reducción de devianza Poisson en lugar de squared error.
            )
        )
    ]
)

### 6.1 Medición del tiempo de entrenamiento

In [13]:
start = perf_counter()

rf_frequency.fit(
    X_train_freq,
    y_train_freq
)

rf_frequency_train_time = (
    perf_counter() - start
)

print(
    f"Tiempo de entrenamiento RF: "
    f"{rf_frequency_train_time:.3f} segundos"
)

Tiempo de entrenamiento RF: 0.619 segundos


### 6.2 Predicciones

In [14]:
pred_train_freq_rf = rf_frequency.predict(
    X_train_freq
)

pred_test_freq_rf = rf_frequency.predict(
    X_test_freq
)

In [ ]:
# Comprobar que no hay predicciones negativas ya que estamos usando criterio Poisson y targets no negativos.
print(
    pred_test_freq_rf.min(),
    pred_test_freq_rf.max()
)

0.015847263513007605 0.7521111385351374


### 6.3 Evaluación inicial del Random Forest de frecuencia

El Random Forest no mejora el desempeño out-of-sample del GLM Poisson.

El modelo obtiene un MAE de 0.2578, un RMSE de 0.3978 y una Poisson deviance
de 0.6224. Estas métricas son ligeramente peores que las obtenidas previamente
por el GLM Poisson.

Por tanto, el incremento de flexibilidad del Random Forest no se traduce,
en esta primera especificación, en una mejora predictiva sobre los conteos
observados.

In [16]:
baseline_freq = np.full(
    len(y_test_freq),
    y_train_freq.mean()
)

In [17]:
rf_freq_mae = mean_absolute_error(
    y_test_freq,
    pred_test_freq_rf
)

rf_freq_rmse = np.sqrt(
    mean_squared_error(
        y_test_freq,
        pred_test_freq_rf
    )
)

rf_freq_deviance = mean_poisson_deviance(
    y_test_freq,
    pred_test_freq_rf
)

print(f"MAE: {rf_freq_mae:.4f}")
print(f"RMSE: {rf_freq_rmse:.4f}")
print(
    f"Poisson deviance: "
    f"{rf_freq_deviance:.4f}"
)

MAE: 0.2578
RMSE: 0.3978
Poisson deviance: 0.6224


### 6.4 Evaluación contra la frecuencia verdadera (*oracle*)

La evaluación contra `lambda_real` revela una diferencia mucho más importante
entre Random Forest y GLM.

El Random Forest obtiene un MAE oracle de 0.0603, un RMSE de 0.0816 y una
correlación de 0.6865 con la frecuencia verdadera.

Estos resultados son considerablemente peores que los obtenidos mediante el
GLM Poisson, cuya correlación oracle era aproximadamente 0.98.

Esto muestra que métricas calculadas únicamente contra los conteos observados
pueden ocultar diferencias importantes entre modelos debido a la elevada
variabilidad aleatoria del proceso Poisson.

En este experimento, el Random Forest está ajustando parcialmente el ruido de
las realizaciones individuales en lugar de recuperar con precisión la función
de frecuencia esperada subyacente.

In [18]:
lambda_true_test = df_test[
    "lambda_real"
]

rf_freq_oracle_mae = (
    mean_absolute_error(
        lambda_true_test,
        pred_test_freq_rf
    )
)

rf_freq_oracle_rmse = np.sqrt(
    mean_squared_error(
        lambda_true_test,
        pred_test_freq_rf
    )
)

rf_freq_oracle_corr = np.corrcoef(
    lambda_true_test,
    pred_test_freq_rf
)[0, 1]

print(
    f"Oracle MAE: "
    f"{rf_freq_oracle_mae:.6f}"
)

print(
    f"Oracle RMSE: "
    f"{rf_freq_oracle_rmse:.6f}"
)

print(
    f"Oracle correlation: "
    f"{rf_freq_oracle_corr:.4f}"
)

Oracle MAE: 0.060336
Oracle RMSE: 0.081628
Oracle correlation: 0.6865


### 6.5 Train vs Test: revisar que no haya overfitting

El desempeño del Random Forest es considerablemente mejor en entrenamiento
que en prueba.

El MAE aumenta de aproximadamente 0.215 en train a 0.258 en test, mientras
que el RMSE pasa de 0.334 a 0.398.

Esta diferencia sugiere que el modelo está capturando parte de la variabilidad
específica de la muestra de entrenamiento y presenta cierto grado de
sobreajuste.

Por este motivo, antes de descartar Random Forest resulta conveniente estudiar
una especificación más regularizada.


In [19]:
rf_train_metrics = regression_metrics(
    y_train_freq,
    pred_train_freq_rf
)

rf_test_metrics = regression_metrics(
    y_test_freq,
    pred_test_freq_rf
)

pd.DataFrame(
    [
        rf_train_metrics,
        rf_test_metrics
    ],
    index=["Train", "Test"]
)

,MAE,RMSE
Train,0.214757,0.333952
Test,0.257800,0.397768


### 6.6 Ajuste de hiperparámetros del Random Forest

El primer Random Forest presentó una diferencia apreciable entre el desempeño
de entrenamiento y prueba, además de una recuperación considerablemente peor
de la frecuencia verdadera que el GLM Poisson.

Antes de comparar definitivamente ambos enfoques, se realizará una búsqueda
moderada de hiperparámetros orientada principalmente a controlar la complejidad
del bosque.

La selección se realizará exclusivamente mediante validación cruzada dentro de
la muestra de entrenamiento. El conjunto de prueba y las cantidades *oracle*
no participan en ninguna decisión de modelado.

#### 6.6.1 Espacio de busqueda

In [ ]:
param_distributions = {
    "model__max_depth": [           # Qué tan complejos pueden ser los árboles (profundidad)
        4,
        6,
        8,
        10,
        None
    ],
    "model__min_samples_leaf": [    # Impide generar hojas sustentadas por muy pocas pólizas
        5,
        10,
        20,
        40
    ],
    "model__max_features": [        # Controla la cantidad de variables que puede considerar cada árbol en cada
        "sqrt",                     # división
        0.7,
        1.0
    ]
}

#### 6.6.2 Scorer de Poisson deviance

In [24]:
from sklearn.metrics import make_scorer
from sklearn.metrics import mean_poisson_deviance


def poisson_deviance_score(y_true, y_pred):
    y_pred = np.clip(
        y_pred,
        1e-10,
        None
    )

    return mean_poisson_deviance(
        y_true,
        y_pred
    )


poisson_scorer = make_scorer(
    poisson_deviance_score,
    greater_is_better=False
)

#### 6.6.3 RandomizedSearchCV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV


rf_search = RandomizedSearchCV(
    estimator=rf_frequency,
    param_distributions=param_distributions,
    n_iter=20,
    scoring=poisson_scorer,
    cv=5,
    refit=True,                     # Entrena el mejor modelo sobre todo df_train después de elegirla
    return_train_score=True,
    random_state=SEED,
    n_jobs=-1,
    verbose=1
)

In [26]:
start = perf_counter()

rf_search.fit(
    X_train_freq,
    y_train_freq
)

rf_search_time = (
    perf_counter() - start
)

print(
    f"Tiempo búsqueda: "
    f"{rf_search_time:.2f} segundos"
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Tiempo búsqueda: 29.08 segundos


#### 6.6.4 Mejor configuración

In [27]:
print(
    "Mejores parámetros:"
)

print(
    rf_search.best_params_
)

print(
    "\nPoisson deviance CV:"
)

print(
    -rf_search.best_score_
)

Mejores parámetros:
{'model__min_samples_leaf': 20, 'model__max_features': 0.7, 'model__max_depth': 4}

Poisson deviance CV:
0.5853769943876005


In [28]:
rf_frequency_tuned = (
    rf_search.best_estimator_
)

#### 6.6.5 Mejores configuraciones

La búsqueda aleatoria seleccionó un Random Forest relativamente poco profundo,
con `max_depth=4`, `min_samples_leaf=20` y `max_features=0.7`.

Esta configuración reduce considerablemente la capacidad del modelo respecto
al bosque inicial.

In [29]:
cv_results = pd.DataFrame(
    rf_search.cv_results_
)

rf_cv_summary = pd.DataFrame({
    "rank": cv_results[
        "rank_test_score"
    ],

    "mean_train_deviance": (
        -cv_results[
            "mean_train_score"
        ]
    ),

    "mean_validation_deviance": (
        -cv_results[
            "mean_test_score"
        ]
    ),

    "std_validation": (
        cv_results[
            "std_test_score"
        ]
    ),

    "max_depth": cv_results[
        "param_model__max_depth"
    ],

    "min_samples_leaf": cv_results[
        "param_model__min_samples_leaf"
    ],

    "max_features": cv_results[
        "param_model__max_features"
    ]
})

rf_cv_summary = (
    rf_cv_summary
    .sort_values("rank")
    .head(10)
)

rf_cv_summary

,rank,mean_train_deviance,mean_validation_deviance,std_validation,max_depth,min_samples_leaf,max_features
17,1,0.567979,0.585377,0.006970,4,20,0.7
1,2,0.567341,0.585619,0.006946,4,10,0.7
19,3,0.566908,0.585684,0.006972,4,5,0.7
11,4,0.524415,0.586046,0.008886,None,20,sqrt
4,5,0.555706,0.586369,0.008125,6,10,sqrt
8,6,0.552908,0.586549,0.008105,6,5,sqrt
12,7,0.539008,0.587072,0.008604,8,40,0.7
15,8,0.539985,0.587572,0.007212,6,10,0.7
16,9,0.565140,0.587586,0.006705,4,5,1.0
0,10,0.576406,0.588435,0.008244,4,5,sqrt


### 6.7 Evaluación sobre test

La regularización elimina gran parte de la diferencia entre entrenamiento y
prueba: el RMSE pasa de aproximadamente 0.389 en train a 0.392 en test.

In [30]:
pred_train_freq_rf_tuned = (
    rf_frequency_tuned.predict(
        X_train_freq
    )
)

pred_test_freq_rf_tuned = (
    rf_frequency_tuned.predict(
        X_test_freq
    )
)

In [31]:
rf_tuned_train_metrics = (
    regression_metrics(
        y_train_freq,
        pred_train_freq_rf_tuned
    )
)

rf_tuned_test_metrics = (
    regression_metrics(
        y_test_freq,
        pred_test_freq_rf_tuned
    )
)

pd.DataFrame(
    [
        rf_tuned_train_metrics,
        rf_tuned_test_metrics
    ],
    index=[
        "Train",
        "Test"
    ]
)

,MAE,RMSE
Train,0.255268,0.389103
Test,0.258924,0.392079


In [32]:
rf_tuned_deviance = (
    mean_poisson_deviance(
        y_test_freq,
        np.clip(
            pred_test_freq_rf_tuned,
            1e-10,
            None
        )
    )
)

print(
    f"Poisson deviance test: "
    f"{rf_tuned_deviance:.4f}"
)

Poisson deviance test: 0.5925


La recuperación de la frecuencia verdadera mejora de manera importante.
La correlación con `lambda_real` aumenta de aproximadamente 0.69 a 0.88 y el
RMSE oracle disminuye de 0.0816 a 0.0375.

Esto sugiere que el Random Forest inicial estaba capturando parcialmente
variabilidad aleatoria de los conteos observados en lugar de la estructura
subyacente de frecuencia.

In [33]:
rf_tuned_oracle_mae = (
    mean_absolute_error(
        lambda_true_test,
        pred_test_freq_rf_tuned
    )
)

rf_tuned_oracle_rmse = np.sqrt(
    mean_squared_error(
        lambda_true_test,
        pred_test_freq_rf_tuned
    )
)

rf_tuned_oracle_corr = np.corrcoef(
    lambda_true_test,
    pred_test_freq_rf_tuned
)[0, 1]


print(
    f"Oracle MAE: "
    f"{rf_tuned_oracle_mae:.6f}"
)

print(
    f"Oracle RMSE: "
    f"{rf_tuned_oracle_rmse:.6f}"
)

print(
    f"Oracle correlation: "
    f"{rf_tuned_oracle_corr:.4f}"
)

Oracle MAE: 0.026815
Oracle RMSE: 0.037533
Oracle correlation: 0.8845


### 6.8 Comparación final: GLM Poisson vs Random Forest

La regularización mejora considerablemente el desempeño del Random Forest,
especialmente en términos de Poisson deviance y recuperación de la frecuencia
verdadera.

Sin embargo, el GLM Poisson continúa mostrando mejores resultados.

Sobre los conteos observados, las diferencias son relativamente pequeñas:
el GLM obtiene menor MAE, RMSE y Poisson deviance.

La diferencia resulta mucho más clara al utilizar el oracle. El GLM alcanza
una correlación de aproximadamente 0.98 con la frecuencia verdadera y un RMSE
oracle de 0.0179, frente a 0.88 y 0.0375 para el Random Forest ajustado.

Por tanto, aunque la regularización permite que Random Forest generalice mucho
mejor, la flexibilidad adicional del modelo no proporciona una ventaja frente
a un GLM cuya estructura funcional se encuentra bien alineada con el problema.

In [34]:
frequency_comparison = pd.DataFrame({
    "Modelo": [
        "GLM Poisson",
        "Random Forest inicial",
        "Random Forest tuned"
    ],

    "MAE test": [
        0.255862,
        rf_freq_mae,
        rf_tuned_test_metrics["MAE"]
    ],

    "RMSE test": [
        0.390971,
        rf_freq_rmse,
        rf_tuned_test_metrics["RMSE"]
    ],

    "Poisson deviance": [
        0.5872,
        rf_freq_deviance,
        rf_tuned_deviance
    ],

    "Oracle MAE": [
        0.013578,
        rf_freq_oracle_mae,
        rf_tuned_oracle_mae
    ],

    "Oracle RMSE": [
        0.017865,
        rf_freq_oracle_rmse,
        rf_tuned_oracle_rmse
    ],

    "Oracle correlation": [
        0.9814,
        rf_freq_oracle_corr,
        rf_tuned_oracle_corr
    ]
})

frequency_comparison

,Modelo,MAE test,RMSE test,Poisson deviance,Oracle MAE,Oracle RMSE,Oracle correlation
0,GLM Poisson,0.255862,0.390971,0.587200,0.013578,0.017865,0.981400
1,Random Forest inicial,0.257800,0.397768,0.622380,0.060336,0.081628,0.686451
2,Random Forest tuned,0.258924,0.392079,0.592515,0.026815,0.037533,0.884533


## 7. XGBoost para frecuencia

Como segundo modelo de Machine Learning se utilizará Gradient Boosting mediante
XGBoost.

A diferencia de Random Forest, donde los árboles se construyen de manera
aproximadamente independiente y posteriormente se promedian, boosting genera
árboles secuencialmente.

Cada nuevo árbol intenta mejorar los errores cometidos por el conjunto de
árboles anteriores.

Para el problema de frecuencia se utiliza una función objetivo Poisson, de
manera que el modelo permanezca alineado con la naturaleza no negativa y de
conteo del target.

### 7.1 Modelo inicial

In [ ]:
xgb_frequency = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            XGBRegressor(
                objective="count:poisson",
                n_estimators=400,
                learning_rate=0.05,     # Hace que cada árbol haga una contribución relativamente pequeña.
                max_depth=3,            # Limita interacciones muy complejas.
                min_child_weight=10,    # Dificulta crear ramas basadas en muy poca información.
                subsample=0.8,          # subsample y colsample_bytree añaden aleatoriedad
                colsample_bytree=0.8,   # para ayudar con generalizacion.
                reg_lambda=1.0,
                random_state=SEED,
                n_jobs=-1,
                tree_method="hist"
            )
        )
    ]
)

### 7.2 Entrenamiento y tiempo

In [36]:
start = perf_counter()

xgb_frequency.fit(
    X_train_freq,
    y_train_freq
)

xgb_frequency_train_time = (
    perf_counter() - start
)

print(
    f"Tiempo entrenamiento XGBoost: "
    f"{xgb_frequency_train_time:.3f} segundos"
)

Tiempo entrenamiento XGBoost: 0.382 segundos


### 7.3 Predicciones

In [37]:
pred_train_freq_xgb = (
    xgb_frequency.predict(
        X_train_freq
    )
)

pred_test_freq_xgb = (
    xgb_frequency.predict(
        X_test_freq
    )
)

In [ ]:
# Con count:poisson deberían ser estrictamente positivas.
print(
    pred_test_freq_xgb.min(),
    pred_test_freq_xgb.max()
)

0.033658996 0.7760566


### 7.4 Métricas observables

El modelo XGBoost presenta un desempeño competitivo respecto al Random Forest
regularizado, aunque todavía no supera al GLM Poisson.

En test obtiene un MAE de 0.2565, un RMSE de 0.3926 y una Poisson deviance de
0.5928. Estas métricas se encuentran muy próximas a las del Random Forest
ajustado y ligeramente por detrás del GLM.

La diferencia entre entrenamiento y prueba es moderada, por lo que no se
observa el sobreajuste pronunciado presentado por el Random Forest inicial.

In [39]:
xgb_freq_train_metrics = regression_metrics(
    y_train_freq,
    pred_train_freq_xgb
)

xgb_freq_test_metrics = regression_metrics(
    y_test_freq,
    pred_test_freq_xgb
)

pd.DataFrame(
    [
        xgb_freq_train_metrics,
        xgb_freq_test_metrics
    ],
    index=[
        "Train",
        "Test"
    ]
)

,MAE,RMSE
Train,0.248632,0.382990
Test,0.256489,0.392573


In [40]:
xgb_freq_deviance = mean_poisson_deviance(
    y_test_freq,
    pred_test_freq_xgb
)

print(
    f"Poisson deviance test: "
    f"{xgb_freq_deviance:.6f}"
)

Poisson deviance test: 0.592776


### 7.5 Evaluación Oracle

En la evaluación oracle, XGBoost alcanza una correlación de 0.91 con la
frecuencia verdadera, superior a la obtenida por Random Forest. Sin embargo,
sus errores absolutos frente al oracle permanecen considerablemente por encima
de los del GLM Poisson.

Esto sugiere que XGBoost recupera razonablemente bien la estructura relativa
del riesgo, aunque todavía presenta errores en la magnitud de la frecuencia
esperada.

In [41]:
xgb_freq_oracle_mae = mean_absolute_error(
    lambda_true_test,
    pred_test_freq_xgb
)

xgb_freq_oracle_rmse = np.sqrt(
    mean_squared_error(
        lambda_true_test,
        pred_test_freq_xgb
    )
)

xgb_freq_oracle_corr = np.corrcoef(
    lambda_true_test,
    pred_test_freq_xgb
)[0, 1]

print(
    f"Oracle MAE: "
    f"{xgb_freq_oracle_mae:.6f}"
)

print(
    f"Oracle RMSE: "
    f"{xgb_freq_oracle_rmse:.6f}"
)

print(
    f"Oracle correlation: "
    f"{xgb_freq_oracle_corr:.4f}"
)

Oracle MAE: 0.027258
Oracle RMSE: 0.038105
Oracle correlation: 0.9102


In [42]:
frequency_comparison_xgb = pd.DataFrame({
    "Modelo": [
        "GLM Poisson",
        "Random Forest tuned",
        "XGBoost inicial"
    ],

    "MAE test": [
        0.255862,
        rf_tuned_test_metrics["MAE"],
        xgb_freq_test_metrics["MAE"]
    ],

    "RMSE test": [
        0.390971,
        rf_tuned_test_metrics["RMSE"],
        xgb_freq_test_metrics["RMSE"]
    ],

    "Poisson deviance": [
        0.5872,
        rf_tuned_deviance,
        xgb_freq_deviance
    ],

    "Oracle MAE": [
        0.013578,
        rf_tuned_oracle_mae,
        xgb_freq_oracle_mae
    ],

    "Oracle RMSE": [
        0.017865,
        rf_tuned_oracle_rmse,
        xgb_freq_oracle_rmse
    ],

    "Oracle correlation": [
        0.9814,
        rf_tuned_oracle_corr,
        xgb_freq_oracle_corr
    ]
})

frequency_comparison_xgb

,Modelo,MAE test,RMSE test,Poisson deviance,Oracle MAE,Oracle RMSE,Oracle correlation
0,GLM Poisson,0.255862,0.390971,0.587200,0.013578,0.017865,0.981400
1,Random Forest tuned,0.258924,0.392079,0.592515,0.026815,0.037533,0.884533
2,XGBoost inicial,0.256489,0.392573,0.592776,0.027258,0.038105,0.910216


### 7.6 Ajuste de hiperpárametros

In [43]:
xgb_param_distributions = {
    "model__n_estimators": [
        200, 400, 600, 800
    ],

    "model__learning_rate": [
        0.02, 0.05, 0.08, 0.10
    ],

    "model__max_depth": [
        2, 3, 4, 5
    ],

    "model__min_child_weight": [
        1, 5, 10, 20
    ],

    "model__subsample": [
        0.7, 0.8, 1.0
    ],

    "model__colsample_bytree": [
        0.7, 0.8, 1.0
    ],

    "model__reg_lambda": [
        1.0, 5.0, 10.0
    ],

    "model__reg_alpha": [
        0.0, 0.1, 0.5
    ]
}

#### 7.6.1 Espacio de busqueda

In [ ]:

xgb_search = RandomizedSearchCV(
    estimator=xgb_frequency,
    param_distributions=xgb_param_distributions,
    n_iter=25,
    scoring=poisson_scorer,
    cv=5,
    refit=True,
    return_train_score=True,
    random_state=SEED,
    n_jobs=-1,
    verbose=1
)

#### 7.6.2 Mejor modelo

La búsqueda mediante validación cruzada seleccionó una configuración
relativamente regularizada, caracterizada por árboles poco profundos
(`max_depth=2`) y una penalización L1/L2 considerable.

In [45]:
start = perf_counter()

xgb_search.fit(
    X_train_freq,
    y_train_freq
)

xgb_search_time = perf_counter() - start

print(
    f"Tiempo búsqueda XGBoost: "
    f"{xgb_search_time:.2f} segundos"
)

print("\nMejores parámetros:")
print(xgb_search.best_params_)

print("\nPoisson deviance CV:")
print(-xgb_search.best_score_)

Fitting 5 folds for each of 25 candidates, totalling 125 fits
Tiempo búsqueda XGBoost: 15.90 segundos

Mejores parámetros:
{'model__subsample': 1.0, 'model__reg_lambda': 10.0, 'model__reg_alpha': 0.5, 'model__n_estimators': 200, 'model__min_child_weight': 1, 'model__max_depth': 2, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.8}

Poisson deviance CV:
0.5837377429008483


In [46]:
xgb_cv_results = pd.DataFrame(
    xgb_search.cv_results_
)

xgb_cv_summary = pd.DataFrame({
    "rank":
        xgb_cv_results["rank_test_score"],

    "train_deviance":
        -xgb_cv_results["mean_train_score"],

    "validation_deviance":
        -xgb_cv_results["mean_test_score"],

    "std_validation":
        xgb_cv_results["std_test_score"],

    "n_estimators":
        xgb_cv_results[
            "param_model__n_estimators"
        ],

    "learning_rate":
        xgb_cv_results[
            "param_model__learning_rate"
        ],

    "max_depth":
        xgb_cv_results[
            "param_model__max_depth"
        ],

    "min_child_weight":
        xgb_cv_results[
            "param_model__min_child_weight"
        ],

    "subsample":
        xgb_cv_results[
            "param_model__subsample"
        ],

    "colsample":
        xgb_cv_results[
            "param_model__colsample_bytree"
        ],

    "reg_lambda":
        xgb_cv_results[
            "param_model__reg_lambda"
        ],

    "reg_alpha":
        xgb_cv_results[
            "param_model__reg_alpha"
        ]
})

xgb_cv_summary = (
    xgb_cv_summary
    .sort_values("rank")
    .head(10)
)

xgb_cv_summary

,rank,train_deviance,validation_deviance,std_validation,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample,reg_lambda,reg_alpha
21,1,0.571148,0.583738,0.007847,200,0.05,2,1,1.0,0.8,10.0,0.5
15,2,0.563192,0.584788,0.008896,200,0.10,2,5,0.8,0.7,1.0,0.5
13,3,0.573033,0.586063,0.008079,200,0.02,3,20,0.8,0.7,5.0,0.1
0,4,0.561230,0.586178,0.008751,400,0.05,2,10,0.7,1.0,1.0,0.0
11,5,0.549943,0.587269,0.007472,200,0.02,5,5,0.7,0.8,10.0,0.0
1,6,0.554608,0.589157,0.008316,800,0.02,3,20,1.0,0.7,5.0,0.5
8,7,0.554157,0.589167,0.008639,600,0.02,3,5,1.0,1.0,5.0,0.0
19,8,0.545770,0.592080,0.008556,200,0.10,3,20,1.0,1.0,5.0,0.0
10,9,0.539532,0.592622,0.009487,400,0.05,3,1,0.7,1.0,5.0,0.1
18,10,0.551537,0.593448,0.008177,600,0.08,2,10,0.7,1.0,1.0,0.5


##### 7.6.3 Evaluación final

Respecto al modelo inicial, el XGBoost ajustado reduce ligeramente el RMSE y la
Poisson deviance en test, aunque presenta un MAE marginalmente mayor.

In [47]:
xgb_frequency_tuned = (
    xgb_search.best_estimator_
)

pred_train_freq_xgb_tuned = (
    xgb_frequency_tuned.predict(
        X_train_freq
    )
)

pred_test_freq_xgb_tuned = (
    xgb_frequency_tuned.predict(
        X_test_freq
    )
)

In [48]:
xgb_tuned_train_metrics = (
    regression_metrics(
        y_train_freq,
        pred_train_freq_xgb_tuned
    )
)

xgb_tuned_test_metrics = (
    regression_metrics(
        y_test_freq,
        pred_test_freq_xgb_tuned
    )
)

pd.DataFrame(
    [
        xgb_tuned_train_metrics,
        xgb_tuned_test_metrics
    ],
    index=[
        "Train",
        "Test"
    ]
)

,MAE,RMSE
Train,0.255527,0.389761
Test,0.258263,0.391349


In [49]:
xgb_tuned_deviance = (
    mean_poisson_deviance(
        y_test_freq,
        pred_test_freq_xgb_tuned
    )
)

print(
    f"Poisson deviance test: "
    f"{xgb_tuned_deviance:.6f}"
)

Poisson deviance test: 0.589806


La mejora resulta más evidente frente al oracle: el RMSE respecto a la
frecuencia verdadera disminuye de aproximadamente 0.0381 a 0.0318 y la
correlación aumenta de 0.91 a 0.92.

Esto indica que la regularización permite recuperar con mayor precisión la
estructura de frecuencia subyacente, evitando parte del ajuste a la variabilidad
aleatoria de los conteos individuales.

In [50]:
xgb_tuned_oracle_mae = (
    mean_absolute_error(
        lambda_true_test,
        pred_test_freq_xgb_tuned
    )
)

xgb_tuned_oracle_rmse = np.sqrt(
    mean_squared_error(
        lambda_true_test,
        pred_test_freq_xgb_tuned
    )
)

xgb_tuned_oracle_corr = np.corrcoef(
    lambda_true_test,
    pred_test_freq_xgb_tuned
)[0, 1]

print(
    f"Oracle MAE: "
    f"{xgb_tuned_oracle_mae:.6f}"
)

print(
    f"Oracle RMSE: "
    f"{xgb_tuned_oracle_rmse:.6f}"
)

print(
    f"Oracle correlation: "
    f"{xgb_tuned_oracle_corr:.4f}"
)

Oracle MAE: 0.022880
Oracle RMSE: 0.031772
Oracle correlation: 0.9212


### Conclusión de los modelos de frecuencia

Los modelos de Machine Learning logran aproximarse al desempeño observable del
GLM Poisson después de aplicar regularización y ajuste de hiperparámetros.

XGBoost presenta el mejor desempeño entre los modelos de árboles y obtiene
métricas out-of-sample muy próximas a las del GLM.

Sin embargo, la evaluación contra la frecuencia verdadera revela una diferencia
más clara. El GLM obtiene un RMSE oracle de aproximadamente 0.018 y una
correlación de 0.981, mientras que XGBoost alcanza aproximadamente 0.032 y
0.921, respectivamente.

Por tanto, dentro de este experimento, la mayor flexibilidad de los modelos de
árboles no proporciona una ventaja frente a un modelo estadístico cuya
especificación funcional se encuentra bien alineada con el proceso generador.

Además, tanto Random Forest como XGBoost requirieron regularización y selección
de hiperparámetros para acercarse al desempeño del GLM, ilustrando que una mayor
complejidad algorítmica no implica automáticamente una mejor recuperación de la
estructura subyacente.

## 8 Random Forest de severidad

In [51]:
X_train_sev = claims_train[FEATURES]
X_test_sev = claims_test[FEATURES]

y_train_sev = claims_train["severidad"]
y_test_sev = claims_test["severidad"]

severity_true_test = claims_test[
    "severidad_esperada_real"
]

### 8.1 Random Forest inicial para severidad

**Random Forest inicial para severidad**

El Random Forest inicial presenta un desempeño out-of-sample próximo, aunque
inferior, al de los modelos estadísticos de severidad.

El modelo obtiene un MAE de aproximadamente $5,714, un RMSE de $9,048 y una
Gamma deviance de 0.3484.

Además, existe una diferencia importante entre entrenamiento y prueba. El RMSE
aumenta de aproximadamente $6,230 en train a $9,048 en test, lo que sugiere
sobreajuste.

La diferencia resulta todavía más clara en la evaluación oracle. El Random
Forest obtiene un RMSE de aproximadamente $2,042 frente a $1,044 del GLM Gamma,
y su correlación con la severidad esperada verdadera disminuye de
aproximadamente 0.983 a 0.926.

Por tanto, la mayor flexibilidad del bosque está capturando parte de la
variabilidad individual de las severidades observadas sin recuperar con la
misma precisión la función de severidad esperada subyacente.

In [52]:
rf_severity = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                max_depth=None,
                min_samples_leaf=5,
                random_state=SEED,
                n_jobs=-1
            )
        )
    ]
)

In [53]:
start = perf_counter()

rf_severity.fit(
    X_train_sev,
    y_train_sev
)

rf_severity_train_time = (
    perf_counter() - start
)

print(
    f"Tiempo RF severidad: "
    f"{rf_severity_train_time:.3f} segundos"
)

Tiempo RF severidad: 0.388 segundos


In [54]:
pred_train_sev_rf = rf_severity.predict(
    X_train_sev
)

pred_test_sev_rf = rf_severity.predict(
    X_test_sev
)

### 8.2 Métricas observables


In [55]:
rf_sev_train_metrics = regression_metrics(
    y_train_sev,
    pred_train_sev_rf
)

rf_sev_test_metrics = regression_metrics(
    y_test_sev,
    pred_test_sev_rf
)

pd.DataFrame(
    [
        rf_sev_train_metrics,
        rf_sev_test_metrics
    ],
    index=["Train", "Test"]
)

,MAE,RMSE
Train,4314.829502,6229.743053
Test,5713.835170,9047.686257


In [56]:
rf_sev_gamma_deviance = (
    mean_gamma_deviance(
        y_test_sev,
        np.clip(
            pred_test_sev_rf,
            1e-10,
            None
        )
    )
)

print(
    f"Gamma deviance test: "
    f"{rf_sev_gamma_deviance:.6f}"
)

Gamma deviance test: 0.348370


### 8.3 Oracle

In [57]:
rf_sev_oracle_mae = mean_absolute_error(
    severity_true_test,
    pred_test_sev_rf
)

rf_sev_oracle_rmse = np.sqrt(
    mean_squared_error(
        severity_true_test,
        pred_test_sev_rf
    )
)

rf_sev_oracle_corr = np.corrcoef(
    severity_true_test,
    pred_test_sev_rf
)[0, 1]

print(
    f"Oracle MAE: "
    f"${rf_sev_oracle_mae:,.2f}"
)

print(
    f"Oracle RMSE: "
    f"${rf_sev_oracle_rmse:,.2f}"
)

print(
    f"Oracle correlation: "
    f"{rf_sev_oracle_corr:.4f}"
)

Oracle MAE: $1,499.41
Oracle RMSE: $2,041.89
Oracle correlation: 0.9264


### 8.4 Randomized Search del Random Forest de Severidad

**Random Forest ajustado para severidad**

La selección de hiperparámetros mediante validación cruzada agrupada por póliza
elige un bosque con profundidad máxima de 6, un mínimo de 5 observaciones por
hoja y selección `sqrt` de variables en cada división.

El ajuste reduce significativamente el sobreajuste observado en el modelo
inicial. Aunque el error de entrenamiento aumenta, el RMSE de prueba disminuye
de aproximadamente $9,048 a $8,926 y la Gamma deviance pasa de 0.348 a 0.325.

La mejora resulta todavía más pronunciada frente al oracle. El RMSE respecto a
la severidad esperada verdadera disminuye de aproximadamente $2,042 a $1,432,
mientras que la correlación aumenta de 0.926 a 0.971.

Por tanto, la regularización permite al Random Forest aproximar con mayor
precisión la función de severidad esperada, aunque su desempeño continúa siendo
inferior al obtenido por los modelos estadísticos Gamma y log-severity.

In [58]:
from sklearn.metrics import make_scorer, mean_gamma_deviance


def gamma_deviance_score(y_true, y_pred):
    y_pred = np.clip(
        y_pred,
        1e-10,
        None
    )

    return mean_gamma_deviance(
        y_true,
        y_pred
    )


gamma_scorer = make_scorer(
    gamma_deviance_score,
    greater_is_better=False
)

In [59]:
# Validación Agrupada para que los siniestros de una misma poliza queden en el mismo grupo
from sklearn.model_selection import GroupKFold

severity_cv = GroupKFold(
    n_splits=5
)

severity_groups = claims_train[
    "policy_id"
]

**Espacio de búsqueda**

In [ ]:
rf_severity_params = {
    "model__criterion": [
        "squared_error",
        "poisson"               # Definimos este criterio solo para cambiar el criterio utilizado, no estamos asumiendo que la severidad sea Poisson
    ],

    "model__max_depth": [
        3,
        4,
        5,
        6,
        8,
        None
    ],

    "model__min_samples_leaf": [
        5,
        10,
        20,
        30,
        40
    ],

    "model__max_features": [
        "sqrt",
        0.7,
        1.0
    ]
}

**Ejecutamos la busqueda**

In [61]:
rf_severity_search = RandomizedSearchCV(
    estimator=rf_severity,
    param_distributions=rf_severity_params,
    n_iter=25,
    scoring=gamma_scorer,
    cv=severity_cv,
    refit=True,
    return_train_score=True,
    random_state=SEED,
    n_jobs=-1,
    verbose=1
)

start = perf_counter()

rf_severity_search.fit(
    X_train_sev,
    y_train_sev,
    groups=severity_groups
)

rf_severity_search_time = (
    perf_counter() - start
)

print(
    f"Tiempo búsqueda RF severidad: "
    f"{rf_severity_search_time:.2f} segundos"
)

print("\nMejores parámetros:")
print(
    rf_severity_search.best_params_
)

print("\nGamma deviance CV:")
print(
    -rf_severity_search.best_score_
)

Fitting 5 folds for each of 25 candidates, totalling 125 fits
Tiempo búsqueda RF severidad: 18.71 segundos

Mejores parámetros:
{'model__min_samples_leaf': 5, 'model__max_features': 'sqrt', 'model__max_depth': 6, 'model__criterion': 'squared_error'}

Gamma deviance CV:
0.3005825724525636


In [62]:
rf_sev_cv_results = pd.DataFrame(
    rf_severity_search.cv_results_
)

rf_sev_cv_summary = pd.DataFrame({
    "rank":
        rf_sev_cv_results["rank_test_score"],

    "train_deviance":
        -rf_sev_cv_results["mean_train_score"],

    "validation_deviance":
        -rf_sev_cv_results["mean_test_score"],

    "std_validation":
        rf_sev_cv_results["std_test_score"],

    "criterion":
        rf_sev_cv_results[
            "param_model__criterion"
        ],

    "max_depth":
        rf_sev_cv_results[
            "param_model__max_depth"
        ],

    "min_samples_leaf":
        rf_sev_cv_results[
            "param_model__min_samples_leaf"
        ],

    "max_features":
        rf_sev_cv_results[
            "param_model__max_features"
        ]
})

(
    rf_sev_cv_summary
    .sort_values("rank")
    .head(10)
)

,rank,train_deviance,validation_deviance,std_validation,criterion,max_depth,min_samples_leaf,max_features
14,1,0.259212,0.300583,0.016820,squared_error,6,5,sqrt
17,2,0.255816,0.302400,0.019453,poisson,5,5,0.7
16,3,0.254526,0.302716,0.018992,squared_error,6,10,0.7
23,4,0.272903,0.303509,0.016768,squared_error,5,5,sqrt
21,5,0.272744,0.303588,0.017085,poisson,5,5,sqrt
13,6,0.270703,0.303815,0.019657,poisson,6,20,0.7
1,7,0.272247,0.303914,0.017950,squared_error,5,20,1.0
19,8,0.249686,0.304612,0.017883,squared_error,6,10,1.0
9,9,0.284946,0.305257,0.018168,poisson,4,30,1.0
7,10,0.282768,0.305963,0.018913,squared_error,8,30,0.7


**Evaluación final**

In [63]:
rf_severity_tuned = (
    rf_severity_search.best_estimator_
)

In [64]:
pred_train_sev_rf_tuned = (
    rf_severity_tuned.predict(
        X_train_sev
    )
)

pred_test_sev_rf_tuned = (
    rf_severity_tuned.predict(
        X_test_sev
    )
)

In [65]:
rf_sev_tuned_train_metrics = (
    regression_metrics(
        y_train_sev,
        pred_train_sev_rf_tuned
    )
)

rf_sev_tuned_test_metrics = (
    regression_metrics(
        y_test_sev,
        pred_test_sev_rf_tuned
    )
)

pd.DataFrame(
    [
        rf_sev_tuned_train_metrics,
        rf_sev_tuned_test_metrics
    ],
    index=["Train", "Test"]
)

,MAE,RMSE
Train,5084.966199,7200.341153
Test,5757.483089,8925.693274


In [66]:
rf_sev_tuned_deviance = (
    mean_gamma_deviance(
        y_test_sev,
        np.clip(
            pred_test_sev_rf_tuned,
            1e-10,
            None
        )
    )
)

print(
    f"Gamma deviance test: "
    f"{rf_sev_tuned_deviance:.6f}"
)

Gamma deviance test: 0.325031


In [67]:
rf_sev_tuned_oracle_mae = (
    mean_absolute_error(
        severity_true_test,
        pred_test_sev_rf_tuned
    )
)

rf_sev_tuned_oracle_rmse = np.sqrt(
    mean_squared_error(
        severity_true_test,
        pred_test_sev_rf_tuned
    )
)

rf_sev_tuned_oracle_corr = np.corrcoef(
    severity_true_test,
    pred_test_sev_rf_tuned
)[0, 1]

print(
    f"Oracle MAE: "
    f"${rf_sev_tuned_oracle_mae:,.2f}"
)

print(
    f"Oracle RMSE: "
    f"${rf_sev_tuned_oracle_rmse:,.2f}"
)

print(
    f"Oracle correlation: "
    f"{rf_sev_tuned_oracle_corr:.4f}"
)

Oracle MAE: $1,024.25
Oracle RMSE: $1,431.76
Oracle correlation: 0.9710


## 9. XGBoost para severidad

Como segundo modelo de Machine Learning para severidad se utilizará XGBoost con
una función objetivo Gamma.

A diferencia del Random Forest, boosting construye árboles secuencialmente,
permitiendo que cada nuevo árbol corrija parte de los errores del conjunto
anterior.

La función objetivo Gamma garantiza además predicciones positivas y resulta
adecuada para una variable continua positiva y asimétrica como la severidad.

### 9.1 Evaluación inicial de XGBoost para severidad

XGBoost obtiene el menor MAE sobre la severidad observada entre los modelos
evaluados hasta este punto, con aproximadamente $5,621 frente a $5,678-$5,689
de los modelos estadísticos.

Sin embargo, no domina en el resto de las métricas. Su RMSE y Gamma deviance
permanecen por encima de los obtenidos por Gamma y log-severity.

La evaluación oracle revela una diferencia todavía más clara. XGBoost obtiene
un RMSE de aproximadamente $1,385 frente a cerca de $1,044 del GLM Gamma.

Por tanto, aunque XGBoost reduce el error absoluto sobre las realizaciones
observadas, todavía recupera con menor precisión la función de severidad
esperada subyacente.

In [68]:
xgb_severity = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            XGBRegressor(
                objective="reg:gamma",
                n_estimators=300,
                learning_rate=0.05,
                max_depth=2,
                min_child_weight=10,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_lambda=5.0,
                reg_alpha=0.1,
                random_state=SEED,
                n_jobs=-1,
                tree_method="hist"
            )
        )
    ]
)

In [69]:
start = perf_counter()

xgb_severity.fit(
    X_train_sev,
    y_train_sev
)

xgb_severity_train_time = (
    perf_counter() - start
)

print(
    f"Tiempo XGBoost severidad: "
    f"{xgb_severity_train_time:.3f} segundos"
)

Tiempo XGBoost severidad: 0.109 segundos


In [70]:
pred_train_sev_xgb = xgb_severity.predict(
    X_train_sev
)

pred_test_sev_xgb = xgb_severity.predict(
    X_test_sev
)

print(
    pred_test_sev_xgb.min(),
    pred_test_sev_xgb.max()
)

5446.263 28183.324


In [71]:
xgb_sev_train_metrics = regression_metrics(
    y_train_sev,
    pred_train_sev_xgb
)

xgb_sev_test_metrics = regression_metrics(
    y_test_sev,
    pred_test_sev_xgb
)

pd.DataFrame(
    [
        xgb_sev_train_metrics,
        xgb_sev_test_metrics
    ],
    index=["Train", "Test"]
)

,MAE,RMSE
Train,5040.856013,7179.484498
Test,5620.641762,8879.727959


In [72]:
xgb_sev_deviance = mean_gamma_deviance(
    y_test_sev,
    pred_test_sev_xgb
)

print(
    f"Gamma deviance test: "
    f"{xgb_sev_deviance:.6f}"
)

Gamma deviance test: 0.322579


In [73]:
xgb_sev_oracle_mae = mean_absolute_error(
    severity_true_test,
    pred_test_sev_xgb
)

xgb_sev_oracle_rmse = np.sqrt(
    mean_squared_error(
        severity_true_test,
        pred_test_sev_xgb
    )
)

xgb_sev_oracle_corr = np.corrcoef(
    severity_true_test,
    pred_test_sev_xgb
)[0, 1]

print(
    f"Oracle MAE: "
    f"${xgb_sev_oracle_mae:,.2f}"
)

print(
    f"Oracle RMSE: "
    f"${xgb_sev_oracle_rmse:,.2f}"
)

print(
    f"Oracle correlation: "
    f"{xgb_sev_oracle_corr:.4f}"
)

Oracle MAE: $960.92
Oracle RMSE: $1,385.16
Oracle correlation: 0.9639


In [74]:
severity_comparison = pd.DataFrame({
    "Modelo": [
        "GLM Gamma",
        "Log-severity + Duan",
        "Random Forest tuned",
        "XGBoost inicial"
    ],
    "MAE test": [
        5689.121329,
        5677.705139,
        rf_sev_tuned_test_metrics["MAE"],
        xgb_sev_test_metrics["MAE"]
    ],
    "RMSE test": [
        8821.826163,
        8807.103748,
        rf_sev_tuned_test_metrics["RMSE"],
        xgb_sev_test_metrics["RMSE"]
    ],
    "Gamma deviance": [
        0.317156,
        0.315653,
        rf_sev_tuned_deviance,
        xgb_sev_deviance
    ],
    "Oracle MAE": [
        696.646,
        705.662,
        rf_sev_tuned_oracle_mae,
        xgb_sev_oracle_mae
    ],
    "Oracle RMSE": [
        1043.530,
        1068.387,
        rf_sev_tuned_oracle_rmse,
        xgb_sev_oracle_rmse
    ],
    "Oracle correlation": [
        0.982558,
        0.981217,
        rf_sev_tuned_oracle_corr,
        xgb_sev_oracle_corr
    ]
})

severity_comparison

,Modelo,MAE test,RMSE test,Gamma deviance,Oracle MAE,Oracle RMSE,Oracle correlation
0,GLM Gamma,5689.121329,8821.826163,0.317156,696.646000,1043.530000,0.982558
1,Log-severity + Duan,5677.705139,8807.103748,0.315653,705.662000,1068.387000,0.981217
2,Random Forest tuned,5757.483089,8925.693274,0.325031,1024.247097,1431.764012,0.971030
3,XGBoost inicial,5620.641762,8879.727959,0.322579,960.923129,1385.158572,0.963861


### 9.2 Randomized Search de XGBoost de Severidad

La validación cruzada selecciona una configuración relativamente regularizada,
con árboles de profundidad máxima 2, 150 estimadores y penalizaciones L1/L2.

Respecto al modelo inicial, el MAE observado aumenta ligeramente, pero el RMSE
y la Gamma deviance mejoran. Esto es consistente con el criterio utilizado
durante la selección de hiperparámetros, basado precisamente en Gamma deviance.

La mejora resulta más evidente frente al oracle. El RMSE respecto a la
severidad esperada verdadera disminuye de aproximadamente $1,385 a $1,285,
mientras que la correlación aumenta de 0.964 a 0.975.

Por tanto, el ajuste de hiperparámetros permite recuperar mejor la función
subyacente de severidad, aunque los modelos estadísticos continúan presentando
una ventaja en la evaluación oracle.

**Área de búsqueda**

In [84]:
xgb_severity_params = {
    "model__n_estimators": [
        150, 300, 500, 700
    ],
    "model__learning_rate": [
        0.02, 0.05, 0.08, 0.10
    ],
    "model__max_depth": [
        2, 3, 4, 5
    ],
    "model__min_child_weight": [
        1, 5, 10, 20
    ],
    "model__subsample": [
        0.7, 0.8, 1.0
    ],
    "model__colsample_bytree": [
        0.7, 0.8, 1.0
    ],
    "model__reg_lambda": [
        1.0, 5.0, 10.0
    ],
    "model__reg_alpha": [
        0.0, 0.1, 0.5
    ]
}

**Búsqueda**

In [85]:
xgb_severity_search = RandomizedSearchCV(
    estimator=xgb_severity,
    param_distributions=xgb_severity_params,
    n_iter=25,
    scoring=gamma_scorer,
    cv=severity_cv,
    refit=True,
    return_train_score=True,
    random_state=SEED,
    n_jobs=-1,
    verbose=1
)

In [86]:
start = perf_counter()

xgb_severity_search.fit(
    X_train_sev,
    y_train_sev,
    groups=severity_groups
)

xgb_severity_search_time = perf_counter() - start

print(
    f"Tiempo búsqueda XGBoost severidad: "
    f"{xgb_severity_search_time:.2f} segundos"
)

print("\nMejores parámetros:")
print(xgb_severity_search.best_params_)

print("\nGamma deviance CV:")
print(-xgb_severity_search.best_score_)

Fitting 5 folds for each of 25 candidates, totalling 125 fits
Tiempo búsqueda XGBoost severidad: 3.49 segundos

Mejores parámetros:
{'model__subsample': 1.0, 'model__reg_lambda': 10.0, 'model__reg_alpha': 0.5, 'model__n_estimators': 150, 'model__min_child_weight': 1, 'model__max_depth': 2, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.8}

Gamma deviance CV:
0.2997283595706588


In [87]:
xgb_sev_cv_results = pd.DataFrame(
    xgb_severity_search.cv_results_
)

xgb_sev_cv_summary = pd.DataFrame({
    "rank":
        xgb_sev_cv_results["rank_test_score"],

    "train_deviance":
        -xgb_sev_cv_results["mean_train_score"],

    "validation_deviance":
        -xgb_sev_cv_results["mean_test_score"],

    "std_validation":
        xgb_sev_cv_results["std_test_score"],

    "n_estimators":
        xgb_sev_cv_results[
            "param_model__n_estimators"
        ],

    "learning_rate":
        xgb_sev_cv_results[
            "param_model__learning_rate"
        ],

    "max_depth":
        xgb_sev_cv_results[
            "param_model__max_depth"
        ],

    "min_child_weight":
        xgb_sev_cv_results[
            "param_model__min_child_weight"
        ],

    "subsample":
        xgb_sev_cv_results[
            "param_model__subsample"
        ],

    "colsample":
        xgb_sev_cv_results[
            "param_model__colsample_bytree"
        ],

    "reg_lambda":
        xgb_sev_cv_results[
            "param_model__reg_lambda"
        ],

    "reg_alpha":
        xgb_sev_cv_results[
            "param_model__reg_alpha"
        ]
})

(
    xgb_sev_cv_summary
    .sort_values("rank")
    .head(10)
)

,rank,train_deviance,validation_deviance,std_validation,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample,reg_lambda,reg_alpha
21,1,0.271427,0.299728,0.017865,150,0.05,2,1,1.0,0.8,10.0,0.5
13,2,0.273657,0.301431,0.018627,150,0.02,3,20,0.8,0.7,5.0,0.1
11,3,0.237375,0.303706,0.020543,150,0.02,5,5,0.7,0.8,10.0,0.0
15,4,0.255770,0.303940,0.019948,150,0.10,2,5,0.8,0.7,1.0,0.5
0,5,0.251426,0.308466,0.023532,300,0.05,2,10,0.7,1.0,1.0,0.0
1,6,0.235606,0.311338,0.021694,700,0.02,3,20,1.0,0.7,5.0,0.5
8,7,0.232643,0.311361,0.023783,500,0.02,3,5,1.0,1.0,5.0,0.0
20,8,0.213982,0.317961,0.026701,150,0.10,4,20,0.7,0.7,10.0,0.5
19,9,0.225986,0.319985,0.024544,150,0.10,3,20,1.0,1.0,5.0,0.0
12,10,0.200637,0.322432,0.026862,500,0.02,5,20,1.0,0.7,1.0,0.1


In [88]:
xgb_severity_tuned = (
    xgb_severity_search.best_estimator_
)

In [89]:
pred_train_sev_xgb_tuned = (
    xgb_severity_tuned.predict(
        X_train_sev
    )
)

pred_test_sev_xgb_tuned = (
    xgb_severity_tuned.predict(
        X_test_sev
    )
)

In [90]:
xgb_sev_tuned_train_metrics = regression_metrics(
    y_train_sev,
    pred_train_sev_xgb_tuned
)

xgb_sev_tuned_test_metrics = regression_metrics(
    y_test_sev,
    pred_test_sev_xgb_tuned
)

pd.DataFrame(
    [
        xgb_sev_tuned_train_metrics,
        xgb_sev_tuned_test_metrics
    ],
    index=["Train", "Test"]
)

,MAE,RMSE
Train,5165.830122,7371.494940
Test,5640.362900,8861.761411


In [81]:
xgb_sev_tuned_deviance = (
    mean_gamma_deviance(
        y_test_sev,
        pred_test_sev_xgb_tuned
    )
)

print(
    f"Gamma deviance test: "
    f"{xgb_sev_tuned_deviance:.6f}"
)

Gamma deviance test: 0.318965


In [82]:
xgb_sev_tuned_oracle_mae = (
    mean_absolute_error(
        severity_true_test,
        pred_test_sev_xgb_tuned
    )
)

xgb_sev_tuned_oracle_rmse = np.sqrt(
    mean_squared_error(
        severity_true_test,
        pred_test_sev_xgb_tuned
    )
)

xgb_sev_tuned_oracle_corr = np.corrcoef(
    severity_true_test,
    pred_test_sev_xgb_tuned
)[0, 1]

print(
    f"Oracle MAE: "
    f"${xgb_sev_tuned_oracle_mae:,.2f}"
)

print(
    f"Oracle RMSE: "
    f"${xgb_sev_tuned_oracle_rmse:,.2f}"
)

print(
    f"Oracle correlation: "
    f"{xgb_sev_tuned_oracle_corr:.4f}"
)

Oracle MAE: $806.40
Oracle RMSE: $1,285.30
Oracle correlation: 0.9751


### Conclusión de los modelos de severidad

Los modelos de Machine Learning consiguen aproximarse considerablemente al
desempeño de los modelos estadísticos de severidad.

XGBoost ajustado obtiene el menor MAE sobre las severidades observadas, aunque
los modelos Gamma y log-severity presentan mejores resultados en RMSE y Gamma
deviance.

La evaluación oracle permite distinguir mejor entre las estrategias. El GLM
Gamma continúa recuperando con mayor precisión la severidad esperada verdadera,
seguido por log-severity y XGBoost.

Por tanto, la mayor flexibilidad de XGBoost proporciona una pequeña ventaja en
error absoluto sobre las realizaciones observadas, pero no mejora la
recuperación de la esperanza condicional subyacente.

## 10. Prima pura mediante Frecuencia × Severidad con modelos de Machine Learning

Los modelos finales de frecuencia y severidad se combinaron para estimar la
prima pura mediante:

$$
\widehat{PP}_i
=
\hat{\lambda}_i
\hat{\mu}_i.
$$

Tanto Random Forest como XGBoost mejoran respecto al baseline que asigna el
mismo costo esperado a todas las pólizas.

Random Forest reduce el MAE aproximadamente 5.1% y el RMSE 3.1%, mientras que
XGBoost consigue reducciones aproximadas de 6.1% y 3.4%, respectivamente.

XGBoost presenta además una Tweedie deviance inferior a la de Random Forest,
indicando un mejor desempeño global sobre el costo agregado.

La diferencia resulta más clara en la evaluación oracle. Random Forest obtiene
un RMSE de aproximadamente $702 y una correlación de 0.918 con la prima pura
verdadera, mientras que XGBoost reduce el RMSE a aproximadamente $598 y aumenta
la correlación a 0.949.

Por tanto, dentro de los enfoques Frecuencia × Severidad basados en Machine
Learning, XGBoost presenta el mejor desempeño.

In [91]:
# Frecuencia: modelos finales ajustados
pred_test_freq_rf_final = rf_frequency_tuned.predict(
    df_test[FEATURES]
)

pred_test_freq_xgb_final = xgb_frequency_tuned.predict(
    df_test[FEATURES]
)

# Severidad esperada para TODAS las pólizas
pred_test_sev_rf_final = rf_severity_tuned.predict(
    df_test[FEATURES]
)

pred_test_sev_xgb_final = xgb_severity_tuned.predict(
    df_test[FEATURES]
)

In [92]:
pred_test_pp_rf = (
    pred_test_freq_rf_final
    * pred_test_sev_rf_final
)

pred_test_pp_xgb = (
    pred_test_freq_xgb_final
    * pred_test_sev_xgb_final
)

In [93]:
pd.DataFrame({
    "RF": pred_test_pp_rf,
    "XGBoost": pred_test_pp_xgb
}).describe()

,RF,XGBoost
count,2000.000000,2000.000000
mean,1867.949918,1835.754517
std,1158.598986,1203.435303
min,478.375346,392.724274
25%,1077.681799,1027.218964
50%,1508.392105,1468.603149
75%,2330.656750,2271.338867
max,9043.544812,9938.059570


**Comparación contra costo observado**

In [94]:
y_test_cost = df_test["costo_siniestros"]

In [101]:
baseline_pp = np.full(
    len(df_test),
    df_train["costo_siniestros"].mean()
)

In [102]:
baseline_pp_metrics_ml = regression_metrics(
    y_test_cost,
    baseline_pp
)

rf_pp_metrics = regression_metrics(
    y_test_cost,
    pred_test_pp_rf
)

xgb_pp_metrics = regression_metrics(
    y_test_cost,
    pred_test_pp_xgb
)

In [103]:
pd.DataFrame(
    [
        baseline_pp_metrics_ml,
        rf_pp_metrics,
        xgb_pp_metrics
    ],
    index=[
        "Baseline media",
        "RF Frecuencia × Severidad",
        "XGBoost Frecuencia × Severidad"
    ]
)

,MAE,RMSE
Baseline media,3380.693184,6650.283158
RF Frecuencia × Severidad,3207.991478,6446.506813
XGBoost Frecuencia × Severidad,3175.127887,6426.867103


In [104]:
EVALUATION_TWEEDIE_POWER = 1.25

In [105]:
baseline_pp_deviance_ml = mean_tweedie_deviance(
    y_test_cost,
    baseline_pp,
    power=EVALUATION_TWEEDIE_POWER
)

rf_pp_deviance = mean_tweedie_deviance(
    y_test_cost,
    pred_test_pp_rf,
    power=EVALUATION_TWEEDIE_POWER
)

xgb_pp_deviance = mean_tweedie_deviance(
    y_test_cost,
    pred_test_pp_xgb,
    power=EVALUATION_TWEEDIE_POWER
)

print(
    f"Baseline:    {baseline_pp_deviance_ml:.4f}"
)

print(
    f"RF F×S:      {rf_pp_deviance:.4f}"
)

print(
    f"XGBoost F×S: {xgb_pp_deviance:.4f}"
)

Baseline:    1320.5237
RF F×S:      1179.8895
XGBoost F×S: 1171.3892


**Oracle**

In [97]:
pp_true_test = df_test[
    "prima_pura_real"
]

In [ ]:
rf_pp_oracle_metrics = regression_metrics(
    pp_true_test,
    pred_test_pp_rf
)

rf_pp_oracle_corr = np.corrcoef(
    pp_true_test,
    pred_test_pp_rf
)[0, 1]

In [ ]:
xgb_pp_oracle_metrics = regression_metrics(
    pp_true_test,
    pred_test_pp_xgb
)

xgb_pp_oracle_corr = np.corrcoef(
    pp_true_test,
    pred_test_pp_xgb
)[0, 1]

In [106]:
ml_pp_oracle_comparison = pd.DataFrame({
    "Modelo": [
        "RF Frecuencia × Severidad",
        "XGBoost Frecuencia × Severidad"
    ],

    "Oracle MAE": [
        rf_pp_oracle_metrics["MAE"],
        xgb_pp_oracle_metrics["MAE"]
    ],

    "Oracle RMSE": [
        rf_pp_oracle_metrics["RMSE"],
        xgb_pp_oracle_metrics["RMSE"]
    ],

    "Oracle correlation": [
        rf_pp_oracle_corr,
        xgb_pp_oracle_corr
    ]
})

ml_pp_oracle_comparison

,Modelo,Oracle MAE,Oracle RMSE,Oracle correlation
0,RF Frecuencia × Severidad,423.432449,701.599529,0.917740
1,XGBoost Frecuencia × Severidad,338.743818,598.304171,0.948913


In [107]:
ml_pp_calibration = pd.DataFrame({
    "Modelo": [
        "RF F×S",
        "XGBoost F×S"
    ],
    "Costo observado medio": [
        y_test_cost.mean(),
        y_test_cost.mean()
    ],
    "Prima predicha media": [
        pred_test_pp_rf.mean(),
        pred_test_pp_xgb.mean()
    ]
})

ml_pp_calibration["Diferencia relativa"] = (
    ml_pp_calibration["Prima predicha media"]
    / ml_pp_calibration["Costo observado medio"]
    - 1
)

ml_pp_calibration

,Modelo,Costo observado medio,Prima predicha media,Diferencia relativa
0,RF F×S,2008.681614,1867.949918,-0.070062
1,XGBoost F×S,2008.681614,1835.754517,-0.086090


## 11. Random Forest para calcular la Prima pura directamente

In [108]:
X_train_cost = df_train[FEATURES]
X_test_cost = df_test[FEATURES]

y_train_cost = df_train["costo_siniestros"]
y_test_cost = df_test["costo_siniestros"]

In [109]:
rf_cost = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                max_depth=6,
                min_samples_leaf=20,
                max_features=0.7,
                random_state=SEED,
                n_jobs=-1,
                criterion="poisson"
            )
        )
    ]
)

In [110]:
start = perf_counter()

rf_cost.fit(
    X_train_cost,
    y_train_cost
)

rf_cost_train_time = perf_counter() - start

print(
    f"Tiempo RF costo directo: "
    f"{rf_cost_train_time:.3f} segundos"
)

Tiempo RF costo directo: 0.437 segundos


In [111]:
pred_train_cost_rf = rf_cost.predict(
    X_train_cost
)

pred_test_cost_rf = rf_cost.predict(
    X_test_cost
)

**Generalización**

El modelo presenta un desempeño mejor en entrenamiento que en prueba, como es
esperable, aunque la diferencia no resulta extrema.

El MAE aumenta de aproximadamente $3,095 a $3,225 y el RMSE de $5,890 a
$6,440.

Esto sugiere cierto deterioro fuera de muestra, pero no un nivel de
sobreajuste comparable al observado en los primeros modelos de árboles de
frecuencia y severidad.

In [112]:
rf_cost_train_metrics = regression_metrics(
    y_train_cost,
    pred_train_cost_rf
)

rf_cost_test_metrics = regression_metrics(
    y_test_cost,
    pred_test_cost_rf
)

pd.DataFrame(
    [
        rf_cost_train_metrics,
        rf_cost_test_metrics
    ],
    index=["Train", "Test"]
)

,MAE,RMSE
Train,3094.598666,5889.969996
Test,3225.278726,6439.621325


**Desempeño sobre el costo observado**

El Random Forest directo mejora respecto al baseline de costo medio, reduciendo
el MAE aproximadamente 4.6% y el RMSE cerca de 3.2%.

Su desempeño es muy similar al del enfoque Random Forest
Frecuencia × Severidad.

El enfoque descompuesto obtiene un MAE ligeramente menor, mientras que el modelo
directo presenta un RMSE marginalmente inferior. Las diferencias son pequeñas,
por lo que no existe una ventaja predictiva clara entre ambas estrategias al
evaluarlas únicamente contra el costo observado.

**Tweedie deviance**

El Random Forest directo obtiene una Tweedie deviance de aproximadamente
1178.51, considerablemente inferior a la del baseline.

El resultado es prácticamente equivalente al obtenido por Random Forest
Frecuencia × Severidad, cuya deviance fue aproximadamente 1179.89.

Aunque el modelo directo presenta el menor valor, la diferencia es demasiado
pequeña para considerarla una ventaja relevante.

In [113]:
rf_cost_deviance = mean_tweedie_deviance(
    y_test_cost,
    np.clip(
        pred_test_cost_rf,
        1e-10,
        None
    ),
    power=EVALUATION_TWEEDIE_POWER
)

print(
    f"Tweedie deviance test: "
    f"{rf_cost_deviance:.4f}"
)

Tweedie deviance test: 1178.5066


**Evaluación contra la prima pura verdadera**

La evaluación oracle permite distinguir mejor entre el enfoque directo y la
estrategia Frecuencia × Severidad.

El Random Forest directo obtiene un MAE de aproximadamente $434, un RMSE de
$718 y una correlación de 0.893 con la prima pura verdadera.

El enfoque Random Forest Frecuencia × Severidad presenta mejores resultados en
las tres métricas, con un MAE cercano a $423, un RMSE de $702 y una correlación
de aproximadamente 0.918.

Esto sugiere que, para Random Forest, descomponer el costo esperado en frecuencia
y severidad proporciona una ligera ventaja en la recuperación de la estructura
real de prima pura.

In [114]:
rf_cost_oracle_metrics = regression_metrics(
    pp_true_test,
    pred_test_cost_rf
)

rf_cost_oracle_corr = np.corrcoef(
    pp_true_test,
    pred_test_cost_rf
)[0, 1]

print(
    f"Oracle MAE: "
    f"${rf_cost_oracle_metrics['MAE']:,.2f}"
)

print(
    f"Oracle RMSE: "
    f"${rf_cost_oracle_metrics['RMSE']:,.2f}"
)

print(
    f"Oracle correlation: "
    f"{rf_cost_oracle_corr:.4f}"
)

Oracle MAE: $433.71
Oracle RMSE: $717.81
Oracle correlation: 0.8930


### Conclusión del Random Forest directo

El modelado directo del costo agregado mediante Random Forest mejora
claramente respecto al baseline y obtiene un desempeño observable muy similar al
enfoque Frecuencia × Severidad.

Sin embargo, la evaluación oracle muestra una ligera ventaja para la estrategia
descompuesta. El modelo Frecuencia × Severidad recupera mejor la prima pura
verdadera tanto en MAE y RMSE como en correlación.

Por tanto, en este experimento, la descomposición de la prima pura parece aportar
estructura útil al Random Forest, aunque la diferencia frente al modelado directo
es moderada.

### 10.1 RandomizedSearchCV para el Random Forest

In [115]:
from sklearn.model_selection import KFold
from sklearn.metrics import make_scorer, mean_tweedie_deviance

cost_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

def tweedie_cost_score(y_true, y_pred):
    y_pred = np.clip(
        y_pred,
        1e-10,
        None
    )

    return mean_tweedie_deviance(
        y_true,
        y_pred,
        power=EVALUATION_TWEEDIE_POWER
    )

tweedie_cost_scorer = make_scorer(
    tweedie_cost_score,
    greater_is_better=False
)

In [116]:
rf_cost_params = {
    "model__criterion": [
        "squared_error",
        "poisson"
    ],

    "model__max_depth": [
        3, 4, 5, 6, 8, 10, None
    ],

    "model__min_samples_leaf": [
        5, 10, 20, 30, 40, 60
    ],

    "model__max_features": [
        "sqrt", 0.5, 0.7, 1.0
    ]
}

In [117]:
rf_cost_search = RandomizedSearchCV(
    estimator=rf_cost,
    param_distributions=rf_cost_params,
    n_iter=30,
    scoring=tweedie_cost_scorer,
    cv=cost_cv,
    refit=True,
    return_train_score=True,
    random_state=SEED,
    n_jobs=-1,
    verbose=1
)

start = perf_counter()

rf_cost_search.fit(
    X_train_cost,
    y_train_cost
)

rf_cost_search_time = perf_counter() - start

print(
    f"Tiempo búsqueda RF costo: "
    f"{rf_cost_search_time:.2f} segundos"
)

print("\nMejores parámetros:")
print(rf_cost_search.best_params_)

print("\nTweedie deviance CV:")
print(-rf_cost_search.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Tiempo búsqueda RF costo: 34.58 segundos

Mejores parámetros:
{'model__min_samples_leaf': 30, 'model__max_features': 0.5, 'model__max_depth': None, 'model__criterion': 'squared_error'}

Tweedie deviance CV:
1161.8287268822348


In [118]:
rf_cost_cv_results = pd.DataFrame(
    rf_cost_search.cv_results_
)

rf_cost_cv_summary = pd.DataFrame({
    "rank":
        rf_cost_cv_results["rank_test_score"],

    "train_deviance":
        -rf_cost_cv_results["mean_train_score"],

    "validation_deviance":
        -rf_cost_cv_results["mean_test_score"],

    "std_validation":
        rf_cost_cv_results["std_test_score"],

    "criterion":
        rf_cost_cv_results[
            "param_model__criterion"
        ],

    "max_depth":
        rf_cost_cv_results[
            "param_model__max_depth"
        ],

    "min_samples_leaf":
        rf_cost_cv_results[
            "param_model__min_samples_leaf"
        ],

    "max_features":
        rf_cost_cv_results[
            "param_model__max_features"
        ]
})

(
    rf_cost_cv_summary
    .sort_values("rank")
    .head(10)
)

,rank,train_deviance,validation_deviance,std_validation,criterion,max_depth,min_samples_leaf,max_features
17,1,1027.234740,1161.828727,33.557997,squared_error,None,30,0.5
2,2,1056.097896,1162.464735,33.347985,poisson,10,40,0.5
29,3,1086.751826,1164.480884,33.090488,squared_error,8,60,0.7
20,4,1080.017767,1164.681703,30.524964,poisson,6,30,0.7
18,5,1117.828785,1166.284253,32.520496,squared_error,8,60,sqrt
1,6,1014.354585,1166.359991,32.568001,squared_error,8,20,0.7
16,7,968.189939,1166.758355,33.139428,squared_error,None,20,0.7
8,8,985.778488,1166.999949,32.667717,poisson,10,20,0.7
22,9,1114.561777,1167.186894,31.783123,squared_error,6,30,sqrt
24,10,1111.750381,1167.540221,30.005116,squared_error,5,30,0.5


**Evaluación final**

In [119]:
rf_cost_tuned = (
    rf_cost_search.best_estimator_
)

pred_train_cost_rf_tuned = (
    rf_cost_tuned.predict(
        X_train_cost
    )
)

pred_test_cost_rf_tuned = (
    rf_cost_tuned.predict(
        X_test_cost
    )
)

In [120]:
rf_cost_tuned_train_metrics = regression_metrics(
    y_train_cost,
    pred_train_cost_rf_tuned
)

rf_cost_tuned_test_metrics = regression_metrics(
    y_test_cost,
    pred_test_cost_rf_tuned
)

pd.DataFrame(
    [
        rf_cost_tuned_train_metrics,
        rf_cost_tuned_test_metrics
    ],
    index=["Train", "Test"]
)

,MAE,RMSE
Train,3055.783596,5856.594153
Test,3226.327926,6438.165607


In [121]:
rf_cost_tuned_deviance = mean_tweedie_deviance(
    y_test_cost,
    np.clip(
        pred_test_cost_rf_tuned,
        1e-10,
        None
    ),
    power=EVALUATION_TWEEDIE_POWER
)

print(
    f"Tweedie deviance test: "
    f"{rf_cost_tuned_deviance:.4f}"
)

Tweedie deviance test: 1184.5414


In [122]:
rf_cost_tuned_oracle_metrics = regression_metrics(
    pp_true_test,
    pred_test_cost_rf_tuned
)

rf_cost_tuned_oracle_corr = np.corrcoef(
    pp_true_test,
    pred_test_cost_rf_tuned
)[0, 1]

print(
    f"Oracle MAE: "
    f"${rf_cost_tuned_oracle_metrics['MAE']:,.2f}"
)

print(
    f"Oracle RMSE: "
    f"${rf_cost_tuned_oracle_metrics['RMSE']:,.2f}"
)

print(
    f"Oracle correlation: "
    f"{rf_cost_tuned_oracle_corr:.4f}"
)

Oracle MAE: $505.28
Oracle RMSE: $743.19
Oracle correlation: 0.8848
